In [34]:
import numpy as np 
import pandas as pd

In [35]:
df = pd.read_csv('diabetes.csv')

In [36]:
df.head(3)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


In [37]:
df.shape

(768, 9)

In [38]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [39]:
df = df.drop(columns=['BloodPressure','SkinThickness'])

In [40]:
from sklearn.model_selection import train_test_split

In [41]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

In [42]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=32)

In [43]:
from sklearn.preprocessing import StandardScaler

In [44]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [49]:
import keras
import tensorflow
from keras.layers import Dense
from keras import Sequential
from tensorflow.keras.layers import Dropout

In [50]:
import keras_tuner as kt

In [51]:
def build_model(hp):

    model = Sequential()

    count= 0 
    for i in range(hp.Int('num_layer',min_value=1,max_value=10,step=1)):
        if count==0:     
            model.add(Dense(
                            units=hp.Int('units'+str(i),
                             min_value=8,
                             max_value=128,
                             step=8),
                             activation=hp.Choice('activation'+str(i),values=['relu','rmsprop','sgd']),
                            input_dim=6
                        ) )
        else:
            model.add(Dense(
                            units=hp.Int('units'+str(i),
                             min_value=8,
                             max_value=128,
                             step=8),
                             activation=hp.Choice('activation'+str(i),values=['relu','sigmoid','tanh']) ) )

        model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8])))
        count += 1 

    model.add(Dense(1,activation="sigmoid"))
    opt = hp.Choice('opt',values=['adam','sgd','rmsprop'])
    model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

    return model
                        

In [52]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,
                       directory='mydir',project_name='final_project')

In [53]:
tuner.search(X_train_scaled,y_train,validation_data=(X_test_scaled,y_test),epochs=5)

Trial 3 Complete [00h 00m 01s]

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 04s


In [54]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 6,
 'units0': 16,
 'activation0': 'relu',
 'dropout0': 0.4,
 'opt': 'rmsprop',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1,
 'units2': 8,
 'activation2': 'relu',
 'dropout2': 0.1,
 'units3': 8,
 'activation3': 'relu',
 'dropout3': 0.1,
 'units4': 8,
 'activation4': 'relu',
 'dropout4': 0.1,
 'units5': 8,
 'activation5': 'relu',
 'dropout5': 0.1}

In [55]:
model = tuner.get_best_models(1)[0]

D:\Deep_Learning\.venv_tf1\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [56]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape               ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 16)                 │             112 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 16)                 │               0 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 8)                  │             136 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 8)                  │               0 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 8)                  │              72 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 8)                  │               0 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 8)                  │              72 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 8)                  │               0 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 8)                  │              72 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 8)                  │               0 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 8)                  │              72 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 8)                  │               0 │
├──────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 1)                  │               9 │
└──────────────────────────────────────┴────────────────────────────┴─────────────────┘

 Total params: 545 (2.13 KB)

 Trainable params: 545 (2.13 KB)

 Non-trainable params: 0 (0.00 B)

In [57]:
model.fit(X_train_scaled,y_train,batch_size=32,epochs=200,initial_epoch=6,validation_data=(X_test_scaled,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6614 - loss: 0.6515 - val_accuracy: 0.6429 - val_loss: 0.6358
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6796 - loss: 0.6256 - val_accuracy: 0.6429 - val_loss: 0.6174
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6390 - loss: 0.6456 - val_accuracy: 0.6429 - val_loss: 0.6074
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6139 - loss: 0.6480 - val_accuracy: 0.6429 - val_loss: 0.5928
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6513 - loss: 0.6033 - val_accuracy: 0.6429 - val_loss: 0.5805
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6477 - loss: 0.6074 - val_accuracy: 0.6429 - val_loss: 0.5690
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6257 - loss: 0.6103 - val_accuracy: 0.6429 - val_loss: 0.5596
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6457 - loss: 0.6062 - val_accuracy: 0.64

In [ ]:
# import shutil
# shutil.rmtree('mydir/final_project') 

### use callback and early stopping to increase more accuracy see it's reached to 82%